# 02 – EDA: thống kê từng trường và trực quan hóa

Notebook chỉ gọi hàm trong `src/eda.py` (issue #8), chạy trên dữ liệu **đã làm sạch** (`clean(load_raw())`). Hình lưu vào `results/figures/`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from src.data import load_raw, label_distribution, PAY_STATUS
from src.preprocess import clean
from src import eda

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
FIG = eda.FIG_DIR

def show(name, width=900):
    display(Image(filename=str(FIG / name), width=width))

## 0. Dữ liệu đã làm sạch

In [2]:
df = clean(load_raw())
print(df.shape)

EDUCATION = 0 -> 4: 14 mẫu
EDUCATION = 5 -> 4: 280 mẫu
EDUCATION = 6 -> 4: 51 mẫu
MARRIAGE = 0 -> 3: 54 mẫu
Loại bản ghi trùng hoàn toàn: 35 dòng (còn 29965)
(29965, 24)


## 1. Thống kê từng cột số

20 cột số: `LIMIT_BAL`, `AGE`, `PAY_1..6`, `BILL_AMT1..6`, `PAY_AMT1..6`. `PAY_x` là biến thứ bậc nhưng mô hình dùng dạng số nên vẫn được tính. Không gồm `SEX`, `EDUCATION`, `MARRIAGE` (mã category) và nhãn. Kurtosis là excess kurtosis (chuẩn = 0).

In [3]:
eda.describe_numeric(df).round(2)

,mean,median,std,min,max,skew,kurtosis
LIMIT_BAL,167442.01,140000.0,129760.14,10000,1000000,0.99,0.54
AGE,35.49,34.0,9.22,21,79,0.73,0.04
PAY_1,-0.02,0.0,1.12,-2,8,0.73,2.73
PAY_2,-0.13,0.0,1.20,-2,8,0.79,1.58
PAY_3,-0.16,0.0,1.20,-2,8,0.84,2.09
PAY_4,-0.22,0.0,1.17,-2,8,1.00,3.51
PAY_5,-0.26,0.0,1.13,-2,8,1.01,4.00
PAY_6,-0.29,0.0,1.15,-2,8,0.95,3.44
BILL_AMT1,51283.01,22438.0,73658.13,-165580,964511,2.66,9.80
BILL_AMT2,49236.37,21295.0,71195.57,-69777,983931,2.70,10.29


**Nhận xét:** các cột tiền lệch phải rất nặng. `BILL_AMT` skew khoảng 2,7–3,1 và `PAY_AMT` skew 10,6–30,4 (kurtosis tới 1639,9 ở `PAY_AMT2`), trung vị thấp hơn nhiều so với trung bình (vd `PAY_AMT1`: 2102 so với 5670). `BILL_AMT` có giá trị âm (min −339603 ở `BILL_AMT6`). `AGE` lệch nhẹ (skew 0,73).

## 2. Bảng tần suất và tỉ lệ vỡ nợ

`ratio` là tỉ trọng của mức trong dữ liệu, `default_rate` là tỉ lệ vỡ nợ trong mức đó.

In [4]:
for c in ['SEX', 'EDUCATION', 'MARRIAGE']:
    print(c)
    display(eda.default_rate_by(df, c).round(3))

SEX


,count,ratio,n_default,default_rate
SEX,,,,
1,11874,0.396,2869,0.242
2,18091,0.604,3761,0.208


EDUCATION


,count,ratio,n_default,default_rate
EDUCATION,,,,
1,10563,0.353,2032,0.192
2,14019,0.468,3328,0.237
3,4915,0.164,1237,0.252
4,468,0.016,33,0.071


MARRIAGE


,count,ratio,n_default,default_rate
MARRIAGE,,,,
1,13643,0.455,3201,0.235
2,15945,0.532,3340,0.209
3,377,0.013,89,0.236


**Nhận xét:** nam vỡ nợ nhiều hơn nữ (24,2% so với 20,8%). Học vấn: THPT (25,2%) và Đại học (23,7%) cao hơn Sau ĐH (19,2%); nhóm Khác (468 mẫu) chỉ 7,1%. Hôn nhân: độc thân 20,9% so với có gia đình 23,5%.

In [5]:
for c in PAY_STATUS:
    print(c)
    display(eda.default_rate_by(df, c).round(3).T)

PAY_1


PAY_1,-2,-1,0,1,2,3,4,5,6,7,8
count,2750.000,5682.000,14737.000,3667.000,2666.000,322.000,76.000,26.000,11.000,9.000,19.000
ratio,0.092,0.190,0.492,0.122,0.089,0.011,0.003,0.001,0.000,0.000,0.001
n_default,364.000,954.000,1888.000,1248.000,1843.000,244.000,52.000,13.000,6.000,7.000,11.000
default_rate,0.132,0.168,0.128,0.340,0.691,0.758,0.684,0.500,0.545,0.778,0.579


PAY_2


PAY_2,-2,-1,0,1,2,3,4,5,6,7,8
count,3752.000,6046.000,15730.000,28.000,3926.000,326.000,99.000,25.000,12.00,20.000,1.0
ratio,0.125,0.202,0.525,0.001,0.131,0.011,0.003,0.001,0.00,0.001,0.0
n_default,686.000,966.000,2503.000,5.000,2183.000,201.000,50.000,15.000,9.00,12.000,0.0
default_rate,0.183,0.160,0.159,0.179,0.556,0.617,0.505,0.600,0.75,0.600,0.0


PAY_3


PAY_3,-2,-1,0,1,2,3,4,5,6,7,8
count,4055.000,5934.000,15764.000,4.00,3819.000,240.000,75.000,21.000,23.000,27.000,3.000
ratio,0.135,0.198,0.526,0.00,0.127,0.008,0.003,0.001,0.001,0.001,0.000
n_default,752.000,926.000,2751.000,1.00,1969.000,138.000,43.000,12.000,14.000,22.000,2.000
default_rate,0.185,0.156,0.175,0.25,0.516,0.575,0.573,0.571,0.609,0.815,0.667


... (2 outputs left out)

PAY_5


PAY_5,-2,-1,0,2,3,4,5,6,7,8
count,4516.000,5535.000,16947.000,2626.000,178.000,83.000,17.000,4.00,58.000,1.0
ratio,0.151,0.185,0.566,0.088,0.006,0.003,0.001,0.00,0.002,0.0
n_default,890.000,897.000,3195.000,1423.000,113.000,50.000,10.000,3.00,48.000,1.0
default_rate,0.197,0.162,0.189,0.542,0.635,0.602,0.588,0.75,0.828,1.0


PAY_6


PAY_6,-2,-1,0,2,3,4,5,6,7,8
count,4865.000,5736.000,16286.000,2766.000,184.000,48.000,13.000,19.000,46.000,2.0
ratio,0.162,0.191,0.544,0.092,0.006,0.002,0.000,0.001,0.002,0.0
n_default,976.000,975.000,3069.000,1401.000,118.000,30.000,7.000,14.000,38.000,2.0
default_rate,0.201,0.170,0.188,0.507,0.641,0.625,0.538,0.737,0.826,1.0


**Nhận xét:** ở mọi `PAY_x`, nhóm trễ từ 2 tháng trở lên có tỉ lệ vỡ nợ rất cao so với nhóm -2/-1/0. Vd `PAY_2 = 2` là 55,6% và `PAY_6 = 2` là 50,7%, trong khi các mức ≤ 0 chỉ 12,8–16,8% ở `PAY_1` và 15,6–20,1% ở `PAY_2..6`. Mức 2 đạt 50,7–69,1% ở cả sáu cột. Các mức ≥ 4 có ít mẫu (tối đa 99, nhiều ô dưới 30, một số chỉ 1–5 mẫu) nên tỉ lệ dao động mạnh. Mức 1 gần như vắng ở `PAY_2..6` (`PAY_2 = 1` chỉ 28 mẫu), trong khi `PAY_1 = 1` có 3667 mẫu.

In [6]:
print('AGE (nhóm tuổi)')
display(eda.default_rate_by(df, 'AGE', bins=eda.AGE_BINS).round(3))
print('LIMIT_BAL (khoảng hạn mức)')
display(eda.default_rate_by(df, 'LIMIT_BAL', bins=eda.LIMIT_BINS).round(3))

AGE (nhóm tuổi)


,count,ratio,n_default,default_rate
AGE,,,,
"(20, 30]",10997,0.367,2470,0.225
"(30, 40]",10702,0.357,2188,0.204
"(40, 50]",5997,0.200,1395,0.233
"(50, 60]",1997,0.067,504,0.252
"(60, 80]",272,0.009,73,0.268


LIMIT_BAL (khoảng hạn mức)


,count,ratio,n_default,default_rate
LIMIT_BAL,,,,
"(0, 50000]",7673,0.256,2439,0.318
"(50000, 100000]",4817,0.161,1244,0.258
"(100000, 150000]",3897,0.130,824,0.211
"(150000, 200000]",3970,0.132,710,0.179
"(200000, 300000]",5053,0.169,811,0.160
"(300000, 500000]",4349,0.145,579,0.133
"(500000, 1000000]",206,0.007,23,0.112


## 3. Tương quan Pearson

In [7]:
eda.plot_correlation_heatmap(df)

WindowsPath('D:/Projects/ml-credit-default/results/figures/correlation_heatmap.png')

![Ma trận tương quan Pearson](../results/figures/correlation_heatmap.png)

**Nhận xét:** 6 cột `BILL_AMT` tương quan rất mạnh với nhau (cặp đôi từ 0,80 đến 0,95), nên mang thông tin gần trùng nhau và gây đa cộng tuyến cho mô hình tuyến tính. Các `PAY_1..6` cũng tương quan dương vừa đến mạnh với nhau (thấp nhất 0,48), còn `PAY_AMT1..6` tương quan yếu với nhau (0,15–0,29) và yếu với `BILL_AMT` (0,10–0,32).

## 4. Năm biểu đồ chọn lọc

### 4.1 Tỉ lệ vỡ nợ theo `PAY_1`
Câu hỏi: lịch sử trễ hạn gần nhất có quyết định khả năng vỡ nợ không?

In [8]:
eda.plot_default_rate_pay1(df)
eda.default_rate_by(df, 'PAY_1').round(3).T

PAY_1,-2,-1,0,1,2,3,4,5,6,7,8
count,2750.000,5682.000,14737.000,3667.000,2666.000,322.000,76.000,26.000,11.000,9.000,19.000
ratio,0.092,0.190,0.492,0.122,0.089,0.011,0.003,0.001,0.000,0.000,0.001
n_default,364.000,954.000,1888.000,1248.000,1843.000,244.000,52.000,13.000,6.000,7.000,11.000
default_rate,0.132,0.168,0.128,0.340,0.691,0.758,0.684,0.500,0.545,0.778,0.579


![Tỉ lệ vỡ nợ theo PAY_1](../results/figures/default_rate_by_pay1.png)

**Nhận xét:** tỉ lệ vỡ nợ nhảy vọt khi bắt đầu trễ hạn: 12,8% ở `PAY_1 = 0` (nhóm lớn nhất, 49,2% mẫu), 34,0% ở trễ 1 tháng và 69,1% ở trễ 2 tháng (2666 mẫu), tới 75,8% ở trễ 3 tháng. Mức -2 và -1 (13,2%, 16,8%) không thấp hơn mức 0, nên `PAY_1` không đơn điệu ở phía ≤ 0. Các mức ≥ 4 ít mẫu (≤ 76) nên không ổn định. `PAY_1` là tín hiệu rất mạnh.

### 4.2 Tỉ lệ vỡ nợ theo khoảng `LIMIT_BAL`
Câu hỏi: hạn mức thấp có rủi ro cao hơn không?

In [9]:
eda.plot_default_rate_limit(df)

WindowsPath('D:/Projects/ml-credit-default/results/figures/default_rate_by_limit_bal.png')

![Tỉ lệ vỡ nợ theo khoảng LIMIT_BAL](../results/figures/default_rate_by_limit_bal.png)

**Nhận xét:** tỉ lệ vỡ nợ giảm đều khi hạn mức tăng: 31,8% ở nhóm ≤ 50k NT$ (25,6% số mẫu), 21,1% ở 100–150k, 13,3% ở 300–500k. Nhóm > 500k chỉ có 206 mẫu, tỉ lệ vỡ nợ 11,2%, nên kém tin cậy. Hạn mức thấp có rủi ro cao hơn rõ rệt.

### 4.3 Tỉ lệ vỡ nợ theo tuổi, học vấn, hôn nhân (và giới tính)
Câu hỏi: hồ sơ nhân khẩu học ảnh hưởng thế nào?

In [10]:
eda.plot_default_rate_demographics(df)

WindowsPath('D:/Projects/ml-credit-default/results/figures/default_rate_by_demographics.png')

![Tỉ lệ vỡ nợ theo nhân khẩu học](../results/figures/default_rate_by_demographics.png)

**Nhận xét:** các yếu tố nhân khẩu học chỉ chênh vài điểm phần trăm quanh mức chung 22,1%, yếu hơn nhiều so với `PAY_1` và `LIMIT_BAL`. Theo tuổi, nhóm 31–40 thấp nhất (20,4%), tăng dần tới 26,8% ở 61–80 (chỉ 272 mẫu). Nhóm học vấn Khác (7,1%) bất thường nhưng chỉ có 468 mẫu.

### 4.4 Xu hướng BILL_AMT và PAY_AMT qua 6 tháng, tách theo nhãn
Câu hỏi: hành vi nhóm vỡ nợ khác nhóm còn lại ra sao theo thời gian? Trục x theo thứ tự thời gian: T4 (chỉ số 6) đến T9/2005 (chỉ số 1).

In [11]:
eda.plot_monthly_trend(df)
print('BILL_AMT'); display(eda.monthly_means(df, 'BILL_AMT').round(0))
print('PAY_AMT'); display(eda.monthly_means(df, 'PAY_AMT').round(0))

BILL_AMT


default,0,1
T4,39091.0,38306.0
T5,40581.0,39576.0
T6,43665.0,42075.0
T7,47592.0,45222.0
T8,49779.0,47326.0
T9,52059.0,48553.0


PAY_AMT


default,0,1
T4,5726.0,3445.0
T5,5255.0,3222.0
T6,5307.0,3158.0
T7,5761.0,3370.0
T8,6649.0,3392.0
T9,6315.0,3400.0


![Xu hướng BILL_AMT và PAY_AMT theo tháng](../results/figures/monthly_trend.png)

**Nhận xét:** sao kê trung bình tăng đều từ T4 đến T9 ở cả hai nhóm; nhóm vỡ nợ có sao kê thấp hơn một chút (T9: 48553 so với 52059). Khác biệt lớn nằm ở số tiền trả: nhóm vỡ nợ trả 3158–3445 mỗi tháng, gần như phẳng, còn nhóm không vỡ nợ trả 5255–6649, tức gấp khoảng 1,6–2,0 lần dù sao kê tương đương. Vậy tiền trả thực tế (so với sao kê) phân biệt hai nhóm tốt hơn bản thân dư nợ.

### 4.5 Phân phối nhãn
Câu hỏi: mức độ mất cân bằng lớp?

In [12]:
eda.plot_label_distribution(df)
label_distribution(df).round(4)

,count,ratio
default,,
0,23335,0.7787
1,6630,0.2213


![Phân phối nhãn](../results/figures/label_distribution.png)

**Nhận xét:** 23335 mẫu không vỡ nợ (77,9%) và 6630 mẫu vỡ nợ (22,1%), tỉ lệ khoảng 3,5 : 1. Mất cân bằng vừa phải: accuracy thuần túy sẽ gây hiểu lầm (luôn đoán 0 đã đạt 77,9%), cần dùng PR-AUC/F1 và xét tăng cường dữ liệu.

## 5. Histogram trước và sau chuẩn hóa (tạm thời)

Chuẩn hóa ở đây là `signed_log` cho cột tiền rồi `StandardScaler` (với `AGE` chỉ scale), **fit trên toàn bộ dữ liệu đã làm sạch, chỉ để mô tả**, không dùng cho huấn luyện vì sẽ rò rỉ. Khi pipeline `build_preprocessor` (#7) xong sẽ thay bằng pipeline đó, fit trên train.

In [13]:
eda.plot_normalization_histograms(df)

WindowsPath('D:/Projects/ml-credit-default/results/figures/hist_before_after_scaling.png')

![Histogram trước và sau chuẩn hóa](../results/figures/hist_before_after_scaling.png)

In [14]:
from src.data import MONEY_COLS
from src.preprocess import signed_log
pd.DataFrame({
    'skew_truoc': df[MONEY_COLS].skew(),
    'skew_sau_log': signed_log(df[MONEY_COLS]).skew(),
}).round(2)

,skew_truoc,skew_sau_log
LIMIT_BAL,0.99,-0.51
BILL_AMT1,2.66,-2.09
BILL_AMT2,2.70,-1.95
BILL_AMT3,3.09,-1.86
BILL_AMT4,2.82,-1.78
BILL_AMT5,2.87,-1.69
BILL_AMT6,2.85,-1.56
PAY_AMT1,14.66,-1.30
PAY_AMT2,30.44,-1.24
PAY_AMT3,17.21,-1.08


In [15]:
from src.data import PAY_AMT, BILL_AMT
zero = pd.DataFrame({
    'ty_le_bang_0': (df[PAY_AMT + BILL_AMT] == 0).mean(),
}).round(3)
zero

,ty_le_bang_0
PAY_AMT1,0.174
PAY_AMT2,0.179
PAY_AMT3,0.198
PAY_AMT4,0.213
PAY_AMT5,0.223
PAY_AMT6,0.238
BILL_AMT1,0.066
BILL_AMT2,0.083
BILL_AMT3,0.095
BILL_AMT4,0.106


**Nhận xét:** trước chuẩn hóa các cột tiền lệch phải nặng (đa số mẫu dồn sát 0). Sau log, `PAY_AMT` giảm skew mạnh (vd `PAY_AMT2` từ 30,4 xuống −1,2) và phần giá trị dương gần hình chuông; `BILL_AMT` chỉ đổi dấu skew (vd `BILL_AMT1` từ 2,7 xuống −2,1). Skew chuyển âm vì còn một cột cao tại giá trị nhỏ nhất: `PAY_AMT1..6` có 17–24% mẫu bằng 0 (17,4% đến 23,8%) và `BILL_AMT1..6` có 6,6–13,3% mẫu bằng 0 (bảng trên), log không làm các cụm này biến mất; riêng `BILL_AMT` còn đuôi âm do sao kê âm. `LIMIT_BAL` vẫn nhiều đỉnh rời rạc do hạn mức thường tròn số. `AGE` chỉ được scale nên hình dạng giữ nguyên.